In [1]:
# NOTEBOOK 44 - calibrated uncertainty for the front end: prediction intervals and breach probability from the final model's REAL errors
# The front end currently draws "80% / 95% intervals" and "breach probability" from a fixed table of sigmas written for the v2 model. This notebook replaces it with
# an empirical calibration of the final model (dyn_adv, real day-old forecast wind):
#   1) predict the VALIDATION windows (2,917) and the TEST windows (2,918) with the 5-seed ensemble
#   2) per forecast horizon, take quantiles of the log-ratio error  z = log((actual + 1) / (forecast + 1))  on the VALIDATION windows only   -> the calibration table
#      (a log-ratio keeps intervals positive and lets them widen with the level, which PM2.5 errors do)
#   3) check it honestly on the TEST windows: coverage of the 80% and 95% intervals per horizon and by forecast level, and the reliability of the breach probability
#      (P that PM2.5 exceeds 60 ug/m3), each against the old fixed-sigma method
# Output: results/dynadv_calibration.json (nothing else is written; make_frontend_data.py copies the table into the front end).
# About 2 minutes on the GPU. LIMIT = a small number tests it quickly.
# HOW TO CHECK: this cell only prints a note.
LIMIT = None
print('calibration for the final model; LIMIT =', LIMIT)

calibration for the final model; LIMIT = None


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
PREVDAY_NPY     = os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy')
ADJ_FC_NPY      = os.path.join(V2_DIR, 'dynamic_adj_prevday1.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')
prevday = np.load(PREVDAY_NPY).astype(np.float32)                    # day-old forecast wind (T,N,2), blowing TOWARD
adj_fc = np.load(ADJ_FC_NPY).astype(np.float32)                      # forecast-wind graph for the future hours (T,N,N)
assert prevday.shape == era5.shape and adj_fc.shape == dyn_adj.shape, (prevday.shape, adj_fc.shape)
print('forecast wind', prevday.shape, '| forecast graph', adj_fc.shape, '| mean |forecast - ERA5| wind speed difference (m/s):',
      float(np.abs(np.hypot(prevday[..., 0], prevday[..., 1]) - era5_spd).mean()))

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508
forecast wind (19921, 37, 2) | forecast graph (19921, 37, 37) | mean |forecast - ERA5| wind speed difference (m/s): 0.7865347862243652


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_14952\1524811802.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F=26  NaNs 0


In [5]:
# 1) geometry and the two new feature pairs for Delhi
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
THRESHOLD_KM, MAX_LAG = 12.0, 6

def geometry(lats, lons):
    n = len(lats)
    dist = np.zeros((n, n)); du = np.zeros((n, n)); dv = np.zeros((n, n))
    R = 6371.0
    for i in range(n):
        for j in range(n):
            if i != j:
                p1, p2 = np.radians(lats[i]), np.radians(lats[j])
                a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lons[j] - lons[i]) / 2) ** 2
                dist[i, j] = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
                de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
                if nr > 0:
                    du[i, j], dv[i, j] = de / nr, dn / nr          # unit vector from i to j
    return dist, du, dv, (dist > 0) & (dist <= THRESHOLD_KM)

def transport_features(pm, uv, dist, du, dv, within):
    # pm (T,n) raw PM2.5, uv (T,n,2) ERA5 wind (blowing TOWARD). Only values at or before hour t are used (no leakage).
    T, n = pm.shape
    adv = pm.copy(); nb = pm.copy()
    src = np.arange(n)[:, None] * np.ones((1, n), dtype=int)           # source station index j for entry [j, i]
    stat = within.astype(float); cnt = stat.sum(0); has_nb = cnt > 0
    for t in range(T):
        align = uv[t, :, 0][:, None] * du + uv[t, :, 1][:, None] * dv   # wind at j along the direction j -> i  (m/s)
        w = np.where(within & (align > 0), align / (dist + 1e-6), 0.0)  # j is upwind of i
        wsum = w.sum(0); has = wsum > 0
        lag = np.clip(np.rint(dist / (np.maximum(align, 0.5) * 3.6)), 0, MAX_LAG).astype(int)   # hours of travel (km / (km/h))
        vals = pm[np.clip(t - lag, 0, T - 1), src]                      # pm[t - lag[j,i], j]
        adv[t, has] = ((w * vals).sum(0))[has] / wsum[has]
        nb[t, has_nb] = ((stat * pm[t][:, None]).sum(0))[has_nb] / cnt[has_nb]
    return adv, nb

dist, du, dv, within = geometry(lats, lons)
t0 = time.time()
adv, nb = transport_features(X_raw[:, :, 0].astype(float), era5, dist, du, dv, within)
pm0 = X_raw[:, :, 0]
extra = np.stack([adv - 0, adv - pm0, nb - 0, nb - pm0], axis=2).astype(np.float32)       # adv_pm, adv_diff, nb_pm, nb_diff
extra[:, :, 0] = adv; extra[:, :, 2] = nb
X30 = np.concatenate([X_raw, extra], axis=2)
print(f'features built in {time.time()-t0:.0f}s | X30 {X30.shape}')
print('mean |adv - own| :', float(np.abs(adv - pm0).mean()), '| mean |nb - own| :', float(np.abs(nb - pm0).mean()))
print('corr(adv, own PM):', float(np.corrcoef(adv.ravel(), pm0.ravel())[0, 1]), '| corr(nb, own PM):', float(np.corrcoef(nb.ravel(), pm0.ravel())[0, 1]))

# HOW TO CHECK: about a minute. Both correlations are high (0.8-0.95) because neighbouring stations resemble each other;
# the question for the models is whether the lagged, wind-aligned version adds anything beyond the same-hour mean.

features built in 1s | X30 (19921, 37, 30)
mean |adv - own| : 28.104649143919474 | mean |nb - own| : 23.36611597159713
corr(adv, own PM): 0.8577609401778713 | corr(nb, own PM): 0.9041709851392609


In [6]:
# 2) scaler for the 30 channels (fit on the training period only); the first 26 must match the saved v3 scaler
from sklearn.preprocessing import StandardScaler
import pickle
T = X30.shape[0]; tr_end = int(T * 0.70)
sc30 = StandardScaler().fit(X30[:tr_end].reshape(-1, 30))
v3s = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
assert np.allclose(sc30.mean_[:26], v3s.mean_, rtol=1e-4, atol=1e-4) and np.allclose(sc30.scale_[:26], v3s.scale_, rtol=1e-4, atol=1e-4), 'scaler mismatch'
log_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Xs_full = sc30.transform(X30.reshape(-1, 30)).reshape(T, N, 30).astype(np.float32)
Yl_full = to_log(X30[:, :, 0])
Y_raw = X30[:, :, 0].copy()
print('scaler check passed: the first 26 channels match the v3 scaler')

# HOW TO CHECK: prints the pass line. An assertion error means the data or train split changed - stop and tell me.

scaler check passed: the first 26 channels match the v3 scaler


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# future-hour wind channels from the day-old forecast (scaled with the same train-period scaler as the observed wind)
pv_spd = np.hypot(prevday[..., 0], prevday[..., 1])
pv_raw = np.stack([prevday[..., 0], prevday[..., 1], pv_spd], axis=2).astype(np.float32)          # (T,N,3)
WIDX = [U_IDX, V_IDX, W_IDX]
Xs_fut = Xs_full.copy()
Xs_fut[:, :, WIDX] = ((pv_raw - sc30.mean_[WIDX]) / sc30.scale_[WIDX]).astype(np.float32)
print('future wind channels now come from the day-old forecast | scaled means: observed', Xs_full[:, :, WIDX].mean((0, 1)).round(2), 'forecast', Xs_fut[:, :, WIDX].mean((0, 1)).round(2))

# HOW TO CHECK: the u and v means are close; the forecast speed mean is lower than the observed one (about -0.5 against -0.1 on the scaled axis) because the day-old forecast runs slower than ERA5. That bias is real and is why the models are retrained on it.

future wind channels now come from the day-old forecast | scaled means: observed [ 0.04  0.01 -0.13] forecast [-0.07  0.14 -0.5 ]


In [8]:
# model with three graph modes plus the learned blend, and optional context dropout at training time
CTX_COLS = ['city_mean_pm', 'city_max_pm', 'pm_minus_city']
CTX_IDX = [FEATURE_COLS.index(c) for c in CTX_COLS]
WSPD_IDX = FEATURE_COLS.index('wspd')
FUT_WSPD = 9                                  # wspd is the last of the 10 future channels (7 calendar + u, v, wspd)

class WATGCNv4(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon, n_stations,
                 dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        sa = sa.clone(); sa.fill_diagonal_(0)
        self.register_buffer('static_adj', sa, persistent=False)
        self.register_buffer('static_rn', sa / sa.sum(1, keepdim=True).clamp(min=1.0), persistent=False)   # row-normalised
        if graph_mode == 'blend':
            self.blend_a = nn.Parameter(torch.zeros(1)); self.blend_b = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def alpha(self, speed):                       # speed: (B, steps) standardised wind speed -> weight of the wind graph
        return torch.sigmoid(self.blend_a + self.blend_b * speed)[..., None, None]

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            A = None
        elif self.graph_mode == 'dynamic':
            A = self._n(At + I.view(1, 1, n, n))
        elif self.graph_mode == 'static':
            A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
        else:                                      # blend
            a = self.alpha(X[..., WSPD_IDX].mean(-1))
            A = self._n(a * At + (1 - a) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(h if A is None else torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        elif self.graph_mode == 'static':
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        else:
            af = self.alpha(Fut[..., FUT_WSPD].mean(-1))
            ctx_fc = torch.matmul(self._n(af * Afc + (1 - af) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n)), ctx_rep)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, self.fut_enc(Fut)], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [9]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

In [10]:
# predictions of the final model (forecast setting) on the validation and test windows
SEEDS = [42, 7, 123, 2024, 999]
ADV_COLS = list(range(26)) + [26, 27]
STATIC_NP = np.load(STATIC_ADJ_NPY).astype(np.float32)
val_end = int(T * 0.85)
def split_starts(lo, hi): return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
STARTS = {'val': split_starts(tr_end, val_end), 'test': split_starts(val_end, T)}
if LIMIT: STARTS = {k: v[:LIMIT] for k, v in STARTS.items()}
class DSE(Dataset):
    def __init__(self, starts, Xfut, Afc): self.s, self.Xfut, self.Afc = np.asarray(starts, np.int64), Xfut, Afc
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(np.ascontiguousarray(Xs_full[g:i1][:, :, ADV_COLS])), torch.from_numpy(dyn_adj[g:i1].copy()),
                torch.from_numpy(self.Afc[i1:i2].copy()), torch.from_numpy(np.ascontiguousarray(self.Xfut[i1:i2][:, :, FUT_IDX])),
                torch.from_numpy(Yl_full[i1 - 1].copy()), torch.from_numpy(Yl_full[i1:i2].copy()), torch.tensor(1.0))
models = []
for sd in SEEDS:
    m = WATGCNv4(len(ADV_COLS), HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, N_FUT, GCN_HOPS, graph_mode='dynamic', static_adj=STATIC_NP).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v5dyn_adv_seed{sd}.pt'), map_location=DEVICE)); m.eval(); models.append(m)
P, A = {}, {}
for split in ('val', 'test'):
    ld = DataLoader(DSE(STARTS[split], Xs_fut, adj_fc), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    outs = [predict(m, ld)[0] for m in models]
    p_log = np.mean(outs, axis=0); a_log = predict(models[0], ld)[1]
    P[split], A[split] = from_log(p_log), from_log(a_log)                                  # (windows, 24, N) ug/m3
    print(f'{split:5} windows {P[split].shape[0]} | RMSE {np.sqrt(np.mean((P[split] - A[split]) ** 2)):.2f} ug/m3')

# HOW TO CHECK: the val RMSE is about 55.05 and the test RMSE about 58.32 (as in notebook 39) when LIMIT is None.

val   windows 2917 | RMSE 55.05 ug/m3
test  windows 2918 | RMSE 58.32 ug/m3


In [11]:
# the calibration table (VALIDATION windows only) and the checks on the TEST windows
LV = [0.5, 1, 2.5, 5, 10, 15, 20, 25, 30, 40, 50, 60, 70, 75, 80, 85, 90, 95, 97.5, 99, 99.5]           # percentile levels stored
def zlog(p, a): return np.log((a + 1.0) / (p + 1.0))
Zv = zlog(P['val'], A['val'])                                                                        # (windows, 24, N)
TABLE = np.stack([np.quantile(Zv[:, h, :].ravel(), np.array(LV) / 100.0) for h in range(HORIZON)])    # (24, len(LV))
lv_idx = {l: i for i, l in enumerate(LV)}

def interval(p, h, lo, hi):                                                                          # calibrated interval for forecast p at horizon index h
    return np.maximum((p + 1.0) * np.exp(TABLE[h, lv_idx[lo]]) - 1.0, 0.0), (p + 1.0) * np.exp(TABLE[h, lv_idx[hi]]) - 1.0

# the old front-end method: fixed sigma per horizon (written for v2), normal intervals
SIGMA = np.array([24.5, 27.2, 29.5, 31.8, 33.4, 34.6, 35.5, 36.3, 37.0, 37.6, 38.1, 38.5, 38.9, 39.3, 39.6, 39.8, 40.1, 40.3, 40.5, 40.6, 40.7, 40.7, 40.6, 40.6])
def old_interval(p, h, z): return np.maximum(p - z * SIGMA[h], 0.0), p + z * SIGMA[h]

pt, at = P['test'], A['test']
print(f"{'horizon':>8}{'80% calibrated':>16}{'80% old':>10}{'95% calibrated':>16}{'95% old':>10}      (share of TEST actuals inside the interval; nominal 80 / 95)")
cov = {'cal80': [], 'old80': [], 'cal95': [], 'old95': []}
for h in range(HORIZON):
    p, a = pt[:, h, :], at[:, h, :]
    l, u = interval(p, h, 10, 90); cov['cal80'].append(float(((a >= l) & (a <= u)).mean()))
    l, u = old_interval(p, h, 1.28); cov['old80'].append(float(((a >= l) & (a <= u)).mean()))
    l, u = interval(p, h, 2.5, 97.5); cov['cal95'].append(float(((a >= l) & (a <= u)).mean()))
    l, u = old_interval(p, h, 1.96); cov['old95'].append(float(((a >= l) & (a <= u)).mean()))
    if h in (0, 2, 5, 11, 17, 23): print(f"{'+' + str(h + 1) + 'h':>8}{100 * cov['cal80'][-1]:>15.1f}%{100 * cov['old80'][-1]:>9.1f}%{100 * cov['cal95'][-1]:>15.1f}%{100 * cov['old95'][-1]:>9.1f}%")
print(f"{'all':>8}{100 * np.mean(cov['cal80']):>15.1f}%{100 * np.mean(cov['old80']):>9.1f}%{100 * np.mean(cov['cal95']):>15.1f}%{100 * np.mean(cov['old95']):>9.1f}%")

# coverage by forecast level (does the interval hold for low, medium and high forecasts?)
edges = np.quantile(pt, [1 / 3, 2 / 3]); bins = np.digitize(pt, edges)
print('\n80% interval coverage on TEST by forecast level (low / middle / high third):')
for nm, fn in (('calibrated', lambda p, h: interval(p, h, 10, 90)), ('old sigma', lambda p, h: old_interval(p, h, 1.28))):
    row = []
    for b in range(3):
        ins = tot = 0
        for h in range(HORIZON):
            l, u = fn(pt[:, h, :], h); mk = bins[:, h, :] == b; ins += int(((at[:, h, :] >= l) & (at[:, h, :] <= u) & mk).sum()); tot += int(mk.sum())
        row.append(100 * ins / max(tot, 1))
    print(f'  {nm:11}' + ''.join(f'{x:>9.1f}%' for x in row))

# breach probability: P(PM2.5 > 60 ug/m3)
THR = 60.0
def breach_prob_cal(p, h):
    z0 = np.log((THR + 1.0) / (p + 1.0)); F = np.interp(z0, TABLE[h], np.array(LV) / 100.0, left=LV[0] / 200.0, right=1 - (100 - LV[-1]) / 200.0)
    return np.clip(1.0 - F, 0.01, 0.99)
def breach_prob_old(p, h):
    from math import erf, sqrt
    z = (THR - p) / SIGMA[h]; return np.clip(1.0 - 0.5 * (1.0 + np.vectorize(erf)(z / sqrt(2.0))), 0.01, 0.99)
y = (at > THR).astype(float)
pc = np.stack([breach_prob_cal(pt[:, h, :], h) for h in range(HORIZON)], axis=1); po = np.stack([breach_prob_old(pt[:, h, :], h) for h in range(HORIZON)], axis=1)
print(f'\nbreach probability on TEST (actual share above {THR:.0f}: {100 * y.mean():.1f}%)  Brier score: calibrated {np.mean((pc - y) ** 2):.4f} | old sigma {np.mean((po - y) ** 2):.4f}  (lower is better)')
print(f"{'predicted prob':>16}{'calibrated: observed':>24}{'old: observed':>16}")
edges_p = np.linspace(0, 1, 6)
for i in range(5):
    mc = (pc >= edges_p[i]) & (pc < edges_p[i + 1] + (1e-9 if i == 4 else 0)); mo = (po >= edges_p[i]) & (po < edges_p[i + 1] + (1e-9 if i == 4 else 0))
    print(f'{f"{100 * edges_p[i]:.0f}-{100 * edges_p[i + 1]:.0f}%":>16}' + (f'{100 * y[mc].mean():>20.1f}% ({int(mc.sum())})' if mc.any() else f'{"-":>24}') + (f'{100 * y[mo].mean():>11.1f}% ({int(mo.sum())})' if mo.any() else f'{"-":>16}'))

out = {'method': 'per-horizon quantiles of z = log((actual + 1) / (forecast + 1)) on the validation windows; interval = (forecast + 1) * exp(q) - 1, lower bound clipped at 0',
       'levels_percent': LV, 'quantiles': [[round(float(x), 4) for x in row] for row in TABLE], 'validation_windows': int(P['val'].shape[0]), 'test_windows': int(P['test'].shape[0]),
       'test_coverage': {k: [round(x, 4) for x in v] for k, v in cov.items()},
       'breach': {'threshold': THR, 'brier_calibrated': float(np.mean((pc - y) ** 2)), 'brier_old_sigma': float(np.mean((po - y) ** 2)), 'observed_share_above': float(y.mean())}}
if not LIMIT: json.dump(out, open(os.path.join(RESULTS_DIR, 'dynadv_calibration.json'), 'w'), indent=1); print('\nsaved results/dynadv_calibration.json')
else: print('\nLIMIT is set: nothing saved')

# HOW TO READ: the calibrated 80% interval should cover close to 80% of TEST actuals at every horizon (the test winter is harder than validation, so a little under is
# expected) and the old sigma interval usually does not; coverage should also be similar for low, middle and high forecasts. In the reliability table, a well calibrated
# probability has an observed share close to its own band. The Brier score should be lower for the calibrated method.

 horizon  80% calibrated   80% old  95% calibrated   95% old      (share of TEST actuals inside the interval; nominal 80 / 95)
     +1h           82.7%     67.2%           96.3%     82.4%
     +3h           82.6%     70.2%           96.6%     84.5%
     +6h           81.9%     72.6%           96.6%     85.8%
    +12h           82.1%     73.7%           96.6%     86.6%
    +18h           82.1%     74.2%           96.6%     86.8%
    +24h           81.8%     73.8%           96.5%     86.2%
     all           82.1%     73.0%           96.5%     86.1%

80% interval coverage on TEST by forecast level (low / middle / high third):
  calibrated      77.0%     83.3%     85.9%
  old sigma       91.7%     76.1%     51.0%

breach probability on TEST (actual share above 60: 77.8%)  Brier score: calibrated 0.0955 | old sigma 0.0965  (lower is better)
  predicted prob    calibrated: observed   old: observed
           0-20%                14.2% (248858)        5.5% (55150)
          20-40%           